# Recommendation Data Preprocess

In [1]:
import os,sys
print(os.getcwd())
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print("Project root directory:", project_root)
sys.path.append(project_root)

/Users/leizhao/projects/meal-rec-ai/preprocess
Project root directory: /Users/leizhao/projects/meal-rec-ai


In [4]:
import pandas as pd
import warnings
from utils.data import root_path
import os
warnings.filterwarnings('ignore')
data_path=f'{root_path}daily_nutrition.csv'
data = pd.read_csv(data_path)
data['daily_food_id'] = data['user_id'].astype(str) + data['years'].astype(str) +  data['day'].astype(int).astype(str)
data = data.drop(columns=['years', 'day'])
pd.set_option('display.max_colwidth', None) 
STANDARD_COLUMNS = ['user_id', 'age',  'gender', 'weight', 'height', 'level',
'under_weight', 'over_weight', 'blood_pressure', 'opioid_misuse', 'diabetes', 'anemia', 'blood_urea_nitrogen', 'osteoporosis', 
'user_low_phosphorus', 'user_low_carb', 'user_low_calorie', 'user_high_calorie', 'user_low_sodium', 'user_high_potassium',  'user_low_saturated_fat', 'user_low_cholesterol', 'low_density_lipoprotein', 'user_low_protein', 'user_high_protein', 'user_low_sugar', 'user_high_fiber','user_high_vitamin_b12', 'user_high_folate_acid', 'user_high_iron', 'user_high_vitamin_c', 'user_high_calcium', 'user_high_vitamin_d', 'grams', 'calorie', 'protein', 'carb', 'sugar', 'fiber', 'saturated_fat', 'cholesterol', 'folic_acid', 'vitamin_b12', 'vitamin_c', 'vitamin_d', 'calcium', 'phosphorus', 'potassium', 'iron', 'sodium','daily_food_id', 'match']
data.columns
original_samples = data[STANDARD_COLUMNS]
original_samples.to_csv(f'{root_path}daily_meal_plan_original.csv',index=False)

In [5]:
len(original_samples)

33348

# Find Negative Samples
- Calculate Gold level daily meal plan
- data[data['match']==True] are all nagtive samples

In [55]:
# 定义函数应用 evaluate 类
from utils.health_evaluate import evaluate_nutrition
from utils.target_calculation import get_nutrition_target_range_by_situation,medical_cols
data=evaluate_nutrition(data)
data=data[data['age']>18]
print(data.columns.tolist())
negative_samples=data[data['match']==False]
print(f'negative_samples daily food:{len(negative_samples)}, \
    user: {len(negative_samples)}')
negative_samples = negative_samples[STANDARD_COLUMNS]
print( negative_samples.columns.tolist())
match_false_count = (negative_samples['match'] == False).sum()
match_false_count==len(negative_samples)
negative_samples.to_csv(f'{root_path}daily_meal_plan_negative.csv',index=False)

['user_id', 'age_group', 'gender', 'grams', 'calorie', 'protein', 'carb', 'sugar', 'fiber', 'saturated_fat', 'cholesterol', 'folic_acid', 'vitamin_b12', 'vitamin_c', 'vitamin_d', 'calcium', 'phosphorus', 'potassium', 'iron', 'sodium', 'age', 'user_low_phosphorus', 'user_low_carb', 'weight', 'height', 'under_weight', 'over_weight', 'user_low_calorie', 'user_high_calorie', 'user_low_sodium', 'user_high_potassium', 'blood_pressure', 'user_low_saturated_fat', 'user_low_cholesterol', 'low_density_lipoprotein', 'blood_urea_nitrogen', 'user_low_protein', 'user_high_protein', 'opioid_misuse', 'diabetes', 'user_low_sugar', 'user_high_fiber', 'anemia', 'user_high_vitamin_b12', 'user_high_folate_acid', 'user_high_iron', 'user_high_vitamin_c', 'user_high_calcium', 'user_high_vitamin_d', 'osteoporosis', 'level', 'b_calorie', 'b_carb', 'b_fiber', 'b_protein', 'b_saturated_fat', 'b_sugar', 'b_cholesterol', 'macro_health_score', 'b_sodium', 'b_phosphorus', 'b_potassium', 'b_iron', 'b_calcium', 'b_foli

## negative_samples: 33200 users ate 33200 daily meal plan out of defined target. 

# Build Positive Samples
merge gold/silve/bronze as unified positive samples

In [75]:
bronze_df = pd.read_csv(f'{root_path}daily_food_with_nutrition_target_bronze.csv')

In [76]:
print("Bronze columns:", bronze_df.columns.tolist())

Bronze columns: ['daily_food_id', 'target', 'user_id', 'grams', 'calorie', 'protein', 'carb', 'sugar', 'fiber', 'saturated_fat', 'cholesterol', 'folic_acid', 'vitamin_b12', 'vitamin_c', 'vitamin_d', 'calcium', 'phosphorus', 'potassium', 'iron', 'sodium', 'age_group', 'gender', 'age', 'user_low_phosphorus', 'user_low_carb', 'weight', 'height', 'under_weight', 'over_weight', 'user_low_calorie', 'user_high_calorie', 'user_low_sodium', 'blood_pressure', 'user_high_potassium', 'user_low_saturated_fat', 'user_low_cholesterol', 'low_density_lipoprotein', 'blood_urea_nitrogen', 'user_low_protein', 'user_high_protein', 'opioid_misuse', 'user_low_sugar', 'user_high_fiber', 'diabetes', 'user_high_folate_acid', 'user_high_iron', 'user_high_vitamin_b12', 'anemia', 'osteoporosis', 'user_high_calcium', 'user_high_vitamin_c', 'user_high_vitamin_d', 'level', 'match']


In [104]:
positive_samples = bronze_df[STANDARD_COLUMNS]
print(f"列: {positive_samples.columns.tolist()}, length:{len(positive_samples)}")

列: ['user_id', 'age', 'gender', 'weight', 'height', 'level', 'under_weight', 'over_weight', 'blood_pressure', 'opioid_misuse', 'diabetes', 'anemia', 'blood_urea_nitrogen', 'osteoporosis', 'user_low_phosphorus', 'user_low_carb', 'user_low_calorie', 'user_high_calorie', 'user_low_sodium', 'user_high_potassium', 'user_low_saturated_fat', 'user_low_cholesterol', 'low_density_lipoprotein', 'user_low_protein', 'user_high_protein', 'user_low_sugar', 'user_high_fiber', 'user_high_vitamin_b12', 'user_high_folate_acid', 'user_high_iron', 'user_high_vitamin_c', 'user_high_calcium', 'user_high_vitamin_d', 'grams', 'calorie', 'protein', 'carb', 'sugar', 'fiber', 'saturated_fat', 'cholesterol', 'folic_acid', 'vitamin_b12', 'vitamin_c', 'vitamin_d', 'calcium', 'phosphorus', 'potassium', 'iron', 'sodium', 'daily_food_id', 'match'], length:214836


In [105]:
positve_samples.to_csv(f'{root_path}daily_meal_plan_positive.csv',index=False)

In [106]:
silver_df = pd.read_csv(f'{root_path}daily_food_with_nutrition_target_silver.csv')

In [107]:
silver_df = silver_df[STANDARD_COLUMNS]
silver_df['match'] = True
print(f"{silver_df.columns.tolist()}")

['user_id', 'age', 'gender', 'weight', 'height', 'level', 'under_weight', 'over_weight', 'blood_pressure', 'opioid_misuse', 'diabetes', 'anemia', 'blood_urea_nitrogen', 'osteoporosis', 'user_low_phosphorus', 'user_low_carb', 'user_low_calorie', 'user_high_calorie', 'user_low_sodium', 'user_high_potassium', 'user_low_saturated_fat', 'user_low_cholesterol', 'low_density_lipoprotein', 'user_low_protein', 'user_high_protein', 'user_low_sugar', 'user_high_fiber', 'user_high_vitamin_b12', 'user_high_folate_acid', 'user_high_iron', 'user_high_vitamin_c', 'user_high_calcium', 'user_high_vitamin_d', 'grams', 'calorie', 'protein', 'carb', 'sugar', 'fiber', 'saturated_fat', 'cholesterol', 'folic_acid', 'vitamin_b12', 'vitamin_c', 'vitamin_d', 'calcium', 'phosphorus', 'potassium', 'iron', 'sodium', 'daily_food_id', 'match']


In [109]:
import pandas as pd

print(f"positive_samples 总行数: {len(positive_samples)}")
print(f"silver 总行数: {len(silver_df)} 行")

# 纵向合并
positive_samples = pd.concat([positive_samples, silver_df], 
                              axis=0, 
                              ignore_index=True)

print(f"\n合并完成！")
print(f"positive_samples 总行数: {len(positive_samples)}")
print(f"其中 silver 贡献: {len(silver_df)} 行")

positive_samples 总行数: 214836
silver 总行数: 11774 行

合并完成！
positive_samples 总行数: 226610
其中 silver 贡献: 11774 行


In [100]:
golden_df = pd.read_csv(f'{root_path}daily_food_with_nutrition_target_gold.csv')

In [102]:
golden_df = golden_df[STANDARD_COLUMNS]
print(f"{golden_df.columns.tolist()}")

['user_id', 'age', 'gender', 'weight', 'height', 'level', 'under_weight', 'over_weight', 'blood_pressure', 'opioid_misuse', 'diabetes', 'anemia', 'blood_urea_nitrogen', 'osteoporosis', 'user_low_phosphorus', 'user_low_carb', 'user_low_calorie', 'user_high_calorie', 'user_low_sodium', 'user_high_potassium', 'user_low_saturated_fat', 'user_low_cholesterol', 'low_density_lipoprotein', 'user_low_protein', 'user_high_protein', 'user_low_sugar', 'user_high_fiber', 'user_high_vitamin_b12', 'user_high_folate_acid', 'user_high_iron', 'user_high_vitamin_c', 'user_high_calcium', 'user_high_vitamin_d', 'grams', 'calorie', 'protein', 'carb', 'sugar', 'fiber', 'saturated_fat', 'cholesterol', 'folic_acid', 'vitamin_b12', 'vitamin_c', 'vitamin_d', 'calcium', 'phosphorus', 'potassium', 'iron', 'sodium', 'daily_food_id', 'match']


In [110]:
print(f"positive_samples 总行数: {len(positive_samples)}")
print(f"golden 总行数: {len(golden_df)} 行")

# 纵向合并
positive_samples = pd.concat([positive_samples, golden_df], 
                              axis=0, 
                              ignore_index=True)

print(f"\n合并完成！")
print(f"positive_samples 总行数: {len(positive_samples)}")
print(f"其中 silver 贡献: {len(golden_df)} 行")


positive_samples 总行数: 226610
golden 总行数: 148 行

合并完成！
positive_samples 总行数: 226758
其中 silver 贡献: 148 行


In [115]:
positive_samples['match']=True
positive_samples.to_csv(f'{root_path}daily_meal_plan_positive.csv',index=False)

In [118]:
match_true_count = (positive_samples['match'] == True).sum()
match_true_count==len(positive_samples)

In [127]:
# 3. 合并数据集
all_data = pd.concat([positive_samples, negative_samples], axis=0, ignore_index=True)

print(f"Positive count: {len(positive_samples)}")
print(f"Negative count: {len(negative_samples)}")
print(f"Total: {len(all_data)}")
# 3. 将 match 转换为数值标签（1/0）
all_data['label'] = all_data['match'].astype(int)

print(f"\n标签分布:")
print(all_data['label'].value_counts())

Positive count: 226758
Negative count: 33200
Total: 259958

标签分布:
label
1    226758
0     33200
Name: count, dtype: int64
